# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
import pandas as pd
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder, FunctionTransformer, PowerTransformer
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, KFold, GridSearchCV  # cross_validate # used for checking, then deleted
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, r2_score

# from sklearn.tree import DecisionTreeClassifier
# from sklearn.metrics import accuracy_score, log_loss, cohen_kappa_score, f1_score

In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt_tot = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
#fires_dt_tot.info()


# Get X and Y

Create the features data frame and target data.

In [3]:
# Create target data frame
df_area_burnt_tot = fires_dt_tot[['area']]
# Create the features data frame
fires_feat_tot = fires_dt_tot.drop(columns=['area'])

In [4]:
#df_area_burnt_tot.describe()

In [5]:
#fires_feat_tot.describe()

Stratified Split Train/Test

In [6]:
# Setting a random seed for GridSearchCV and cross_validate later used, even though we finally found how to embed random_state inside those objects, I left it in case I will need it
random.seed(42)

# Assuming `fires_feat` is your feature set and `df_area_burnt` is your target
stratify_labels = (df_area_burnt_tot == 0).astype(int)  # Convert target into binary labels

fires_feat_train, fires_feat_test, df_area_burnt_train, df_area_burnt_test = train_test_split(
    fires_feat_tot, df_area_burnt_tot, test_size=0.2, stratify=stratify_labels, random_state=42
)

#print(f"Train target distribution: {df_area_burnt_train.value_counts(normalize=True)}")
#print(f"Test target distribution: {df_area_burnt_test.value_counts(normalize=True)}")

# to shorten variable names
fires_feat = fires_feat_train.copy()
df_area_burnt = df_area_burnt_train.copy()
fires_dt = pd.concat([fires_feat,df_area_burnt], axis=1)

Non-Stratified Split Train/Test. Disabled

In [7]:
# # Setting a random seed for GridSearchCV and cross_validate later used, even though we finally found how to embed random_state inside those objects
# random.seed(42)
# # Splitting Strategy: Create training and test datasets
# fires_feat_train, fires_feat_test, df_area_burnt_train, df_area_burnt_test = train_test_split(fires_feat, df_area_burnt, test_size=0.2, random_state=42)
# #print(fires_feat_train.shape, fires_feat_test.shape, df_area_burnt_train.shape, df_area_burnt_test.shape)
# # to shorten variable names
# fires_feat = fires_feat_train.copy()
# df_area_burnt = df_area_burnt_train.copy()
# fires_dt = pd.concat([fires_feat,df_area_burnt], axis=1)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

Remove outliers. We will do a logarithmic transformation of Area before that.

In [8]:
# To remove some extreme outliers, we will use Interquartile Range (IQR), with outliers defined as values below Q1 - 2.5 * IQR or above Q3 + 2.5 * IQR. Very soft removal.
def outlier_removal(fires_dt, train_or_test):
    # Let's apply log transformation of variable area burnt
    # Adding a small constant to avoid log(0)
    fires_dt.loc[:,'log_area'] = np.log(fires_dt['area'] + 1)  # Using +epsilon ensures no zero issues # 12.84729 is average area burnt (includes zeros) # 17.61817 is average area burnt (excludes zeros) 
    # Now let's remove just few outliers. We need to check outliers on numeric features only but remove the rows from both numeric and categorical features and target variable (all columns)
    if train_or_test == 'train':    # when fitting train, I want to remove an area outlier, it could be a typo, oversight or just very rare occurrence
        fires_dt_look_for_outliers = fires_dt[['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'log_area']]
    elif train_or_test == 'test':   # when fitting test, it is a simulation of reality, I don't know know what will be the outcome, area is what I want to predict; but I know my measured features, I can always chose not to predict
        fires_dt_look_for_outliers = fires_dt[['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind']]
    # Compute Q1 (25th percentile) and Q3 (75th percentile) for each column
    Q1 = fires_dt_look_for_outliers.quantile(0.25)
    Q3 = fires_dt_look_for_outliers.quantile(0.75)
    IQR = Q3 - Q1
    # Define outlier thresholds
    lower_bound = Q1 - 2.5 * IQR
    upper_bound = Q3 + 2.5 * IQR
    # Find rows where any column has an outlier
    outlier_rows = ((fires_dt_look_for_outliers < lower_bound) | (fires_dt_look_for_outliers > upper_bound)).any(axis=1)
    # Remove rows that contain any outlier
    fires_dt_noutlier = fires_dt[~outlier_rows]
    return fires_dt_noutlier

fires_dt_noutlier = outlier_removal(fires_dt,'train')
#print(fires_dt_noutlier.shape, fires_dt_noutlier.columns)
# Objects we will use to fit the pipeline thereafter if we wish to remove outliers. All the preprocessing below will happen inside the pipelines, except remove outliers
fires_feat_noutlier = fires_dt_noutlier.drop(columns=['area','log_area'])
df_area_burnt_noutlier = fires_dt_noutlier[['area']] # does not contain log_area because pipeline will deal with log_area itself
# Objects used in preprocessing below in this section (outside of the pipeline)
fires_logarea_noutlier = fires_dt_noutlier[['log_area']]
fires_num_cat_noutlier = fires_dt_noutlier.drop(columns='log_area')
#print(fires_num_cat_noutlier.shape, fires_num_cat_noutlier.columns)
#print(type(fires_logarea_noutlier))
#print(fires_logarea_noutlier.shape, fires_logarea_noutlier.columns)

In [9]:
# TO BE SELECTED ONE BLOCK ONLY
##########################################################
# BLOCK: Without outliers removal - With outliers
# X2_train = fires_feat_train.copy()      # shape (413, 12)
# Y2_train = df_area_burnt_train.copy()   # shape (413, 1) 
##########################################################
# BLOCK: With outliers removal - Without some outliers
X2_train = fires_feat_noutlier.copy()     # shape (393, 12)
Y2_train = df_area_burnt_noutlier.copy()  # shape (393, 1) 
##########################################################
# print(X2_train.shape, X2_train.columns)
# print(Y2_train.shape, Y2_train.columns)

Let's create the transformer that will use the next section

In [10]:
# Custom transformer to convert numeric to categorical (example threshold-based binning)
def numeric_to_category(X):
    series = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else X.ravel()
    return np.where(series == 0, 'dried', 'rained').reshape(-1, 1)

numeric_to_category_transformer = FunctionTransformer(numeric_to_category)

# Build numeric pipeline
pipe_numeric_skewed =   Pipeline([
                        ('transform', PowerTransformer(method='yeo-johnson')),
                        ('standardizer', StandardScaler())

                        ])
# Building rain pipeline
pipe_rain = Pipeline([
            ('convert_rain', numeric_to_category_transformer),
            ('onehot', OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist'))
            ])

# Build transformer
transformer2 = ColumnTransformer(
    transformers=[
        # Apply standard scaling to some numerical features
        ('numeric', StandardScaler(), ['ffmc', 'dmc', 'dc', 'isi', 'wind']),
        # Apply standard scaling and yeo-johnson to numerical features temp and rh
        ('numeric_skewed', pipe_numeric_skewed, ['temp', 'rh']),
        # OneHotEncode month and day (no transformation needed)
        ('onehot_nominal', OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist'), ['coord_x', 'coord_y', 'month', 'day']),        
        # Convert rain and then one-hot encode
        ('rain', pipe_rain, ['rain'])
    ],
    remainder ='drop' 
)

# ['coord_x', 'coord_y'] not included among numeric features because those do not reflect low or high values, just different location
transformer2

ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['ffmc', 'dmc', 'dc', 'isi', 'wind']),
                                ('numeric_skewed',
                                 Pipeline(steps=[('transform',
                                                  PowerTransformer()),
                                                 ('standardizer',
                                                  StandardScaler())]),
                                 ['temp', 'rh']),
                                ('onehot_nominal',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='infrequent_if_exist'),
                                 ['coord_x', 'coord_y', 'month', 'day']),
                                ('rain',
                                 Pipeline(steps=[('convert_rain',
                                                  FunctionTransformer(func=<function numeric_to_category at 0x0000024D4A3CBEE0>)),
                                                 ('onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='infrequent_if_exist'))]),
                                 ['rain'])])

Let's summarize what we did in Preproc 2, on the training data only:
- Convert rain from numerical to categorical (named rained)
- Apply log transformation of target variable area burnt (ln + 0.001)
- Remove extreme outliers (Q1,Q3 +- 2.5 IQR) found in numerical variables (including target variable area burnt) from all features (numerical and categorical), less than 6% of the sample
- Standardize all numerical features except target variable area burnt
- One-Hot Encoding in categorical features, including rained

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [11]:
# Create a binary target: is_not_zero = 1 if y != 0 else 0
y_train_binary = (Y2_train != 0).astype(int).values.ravel()  # This will be your classification target

# Classification pipeline
pipe_classification = Pipeline([
    ('preprocessing', transformer2),
    ('model', LogisticRegression())
])
pipe_classification

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['ffmc', 'dmc', 'dc', 'isi',
                                                   'wind']),
                                                 ('numeric_skewed',
                                                  Pipeline(steps=[('transform',
                                                                   PowerTransformer()),
                                                                  ('standardizer',
                                                                   StandardScaler())]),
                                                  ['temp', 'rh']),
                                                 ('onehot_nominal',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='infrequent_if_exist'),
                                                  ['coord_x', 'coord_y',
                                                   'month', 'day']),
                                                 ('rain',
                                                  Pipeline(steps=[('convert_rain',
                                                                   FunctionTransformer(func=<function numeric_to_category at 0x0000024D4A3CBEE0>)),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='infrequent_if_exist'))]),
                                                  ['rain'])])),
                ('model', LogisticRegression())])

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

Pipeline F

In [12]:
param_grid = {
    'model__solver': ['lbfgs'],#['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'], #['lbfgs'],#
    'model__C': [0.01],#[0.01, 0.1, 1, 10, 100, 1000],  # Regularization strength [0.01],#
    'model__penalty': ['l2']#['l1', 'l2','elasticnet']  # Some only work with some solvers, it fails but does not give error, still checks all  ['l2']#
}
#Try:
# 0.01	l2	lbfgs	    done best
# 0.10	l1	saga	    done
# 0.01	l2	liblinear  	done
# 0.01	l2	saga	    done
# 0.01	l2	sag	        done best

cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)        # 5-fold cross-validation
scoring = {
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'f1': 'f1',
    'recall': 'recall'
}
grid_search = GridSearchCV(
    estimator=pipe_classification,
    param_grid=param_grid,
    scoring = scoring,
    refit='f1',      # or 'accuracy', 'f1', etc.
    cv=cv_strategy,           
    n_jobs=-1               # use all cores
)

grid_search.fit(X2_train, y_train_binary)

GridSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         StandardScaler(),
                                                                         ['ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'wind']),
                                                                        ('numeric_skewed',
                                                                         Pipeline(steps=[('transform',
                                                                                          PowerTransformer()),
                                                                                         ('standardizer',
                                                                                          StandardScaler())]),
                                                                         ['temp',
                                                                          'rh']),
                                                                        ('onehot_nominal',
                                                                         OneHotEncoder(...
                                                                                          FunctionTransformer(func=<function numeric_to_category at 0x0000024D4A3CBEE0>)),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(drop='first',
                                                                                                        handle_unknown='infrequent_if_exist'))]),
                                                                         ['rain'])])),
                                       ('model', LogisticRegression())]),
             n_jobs=-1,
             param_grid={'model__C': [0.01], 'model__penalty': ['l2'],
                         'model__solver': ['lbfgs']},
             refit='f1',
             scoring={'accuracy': 'accuracy', 'f1': 'f1', 'recall': 'recall',
                      'roc_auc': 'roc_auc'})

In [13]:
print("Best score (CV average):", grid_search.best_score_)


Best score (CV average): 0.6529059215344002


In [14]:
print("Best parameters:", grid_search.best_params_)


Best parameters: {'model__C': 0.01, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}


In [15]:
best_model_classification = grid_search.best_estimator_


In [16]:
import pandas as pd

cv_results_df = pd.DataFrame(grid_search.cv_results_)
#cv_results_df.sort_values(by='mean_test_score', ascending=False)#.head()
a=cv_results_df[['param_model__C', 'param_model__penalty', 'param_model__solver', 'mean_test_roc_auc', 'mean_test_accuracy', 'mean_test_f1', 'mean_test_recall', 'rank_test_roc_auc', 'rank_test_accuracy', 'rank_test_f1','rank_test_recall']]
a.sort_values(by='rank_test_f1', ascending=True).head()

,param_model__C,param_model__penalty,param_model__solver,mean_test_roc_auc,mean_test_accuracy,mean_test_f1,mean_test_recall,rank_test_roc_auc,rank_test_accuracy,rank_test_f1,rank_test_recall
0,0.01,l2,lbfgs,0.531845,0.542129,0.652906,0.822119,1,1,1,1


In [17]:
y_class_pred = best_model_classification.predict(X2_train)

# Count how many were predicted as each class
import numpy as np
unique, counts = np.unique(y_class_pred, return_counts=True)
print(dict(zip(unique, counts)))  # e.g., {0: 480, 1: 520}  # {0: 99, 1: 314} (no outliers removal) # {0: 76, 1: 317} {0: 83, 1: 310}


{0: 80, 1: 313}


In [18]:
# y_proba_train = best_model.predict_proba(X2_train)[:, 1]  # Probability of class 1 (non-zero)
# threshold = 0.6
# y_pred_custom = (y_proba_train > threshold).astype(int)
# y_pred_custom


## Model Pipeline

In [19]:
# Pipeline F = preproc2 + baseline

pipe_regression_pre = Pipeline(
    [
        ('preprocessing', transformer2), 
        ('poly', PolynomialFeatures()),
        ('model', LinearRegression())  
    ]
)

pipe_regression =  TransformedTargetRegressor(regressor=pipe_regression_pre, func=np.log1p, inverse_func=np.expm1)   # Apply log transform to target

In [20]:
# Subset for regression: non-zero targets only
mask_non_zero = y_class_pred == 1
X_non_zero = X2_train.loc[mask_non_zero].copy()
y_non_zero = Y2_train.loc[mask_non_zero].copy()
# Fit the pipeline
pipe_regression.fit(X_non_zero, y_non_zero)

TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('preprocessing',
                                                      ColumnTransformer(transformers=[('numeric',
                                                                                       StandardScaler(),
                                                                                       ['ffmc',
                                                                                        'dmc',
                                                                                        'dc',
                                                                                        'isi',
                                                                                        'wind']),
                                                                                      ('numeric_skewed',
                                                                                       Pipeline(steps=[('transform',
                                                                                                        PowerTransformer()),
                                                                                                       ('standardizer',
                                                                                                        StandardScaler())]),
                                                                                       ['temp',
                                                                                        'rh']),
                                                                                      ('onehot_nominal',
                                                                                       On...irst',
                                                                                                     handle_unknown='infrequent_if_exist'),
                                                                                       ['coord_x',
                                                                                        'coord_y',
                                                                                        'month',
                                                                                        'day']),
                                                                                      ('rain',
                                                                                       Pipeline(steps=[('convert_rain',
                                                                                                        FunctionTransformer(func=<function numeric_to_category at 0x0000024D4A3CBEE0>)),
                                                                                                       ('onehot',
                                                                                                        OneHotEncoder(drop='first',
                                                                                                                      handle_unknown='infrequent_if_exist'))]),
                                                                                       ['rain'])])),
                                                     ('poly',
                                                      PolynomialFeatures()),
                                                     ('model',
                                                      LinearRegression())]))

# Tune Hyperparams

In [21]:
param_grid = {
    'regressor__poly__degree': [1, 2, 3, 4],
    'regressor__poly__interaction_only': [False, True],
    'regressor__poly__include_bias': [False],
    'regressor__model__fit_intercept': [True, False]
}
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'neg_rmse': 'neg_root_mean_squared_error',
    'r2': 'r2'
}
grid_search_F = GridSearchCV(pipe_regression, param_grid, scoring=scoring, refit='neg_rmse', cv=cv_strategy, return_train_score = True) 
grid_search_F.fit(X_non_zero, y_non_zero)

print("Best parameters:", grid_search_F.best_params_)
# refit='neg_mse'  # Best parameters: {'model__fit_intercept': False, 'poly__degree': 1, 'poly__include_bias': False, 'poly__interaction_only': False}

c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\miniconda3\envs\dsi_participant\lib\site-packages\sklearn

Best parameters: {'regressor__model__fit_intercept': True, 'regressor__poly__degree': 1, 'regressor__poly__include_bias': False, 'regressor__poly__interaction_only': False}


In [22]:
# Get the best pipeline from GridSearchCV, which is the fit to the whole trainiing set with the best grid parameters
best_pipeline_regression_F = grid_search_F.best_estimator_
# Predict regression values
y_reg_pred = best_pipeline_regression_F.predict(X_non_zero)
print(y_reg_pred.shape)

(313, 1)


In [23]:
print(Y2_train.shape)
# Step 3: Initialize full prediction array
y_full_pred = np.zeros_like(Y2_train)
# Step 4: Insert regression predictions into predicted non-zero locations
y_full_pred[mask_non_zero] = y_reg_pred
print(mask_non_zero.shape)
print(y_reg_pred.shape)
print(y_full_pred.shape)


(393, 1)
(393,)
(313, 1)
(393, 1)


In [24]:
# Step 5: Evaluate training dataset
rmse = root_mean_squared_error(Y2_train, y_full_pred)
r2 = r2_score(Y2_train, y_full_pred)

print(f"Combined RMSE: {rmse:.2f}")     # Combined RMSE: 43.61
print(f"Combined R²: {r2:.2f}")         # Combined R²: 0.02

Combined RMSE: 43.61
Combined R²: 0.02


In [25]:
# RMSE and R2 of BestPipeline on Test dataset

# TO BE SELECTED ONE BLOCK ONLY (SAME BLOCK AS ABOVE)
##########################################################
# # BLOCK: Without outliers removal - With outliers
# Recover TEST dataset
# X1_test = fires_feat_test.copy()
# Y1_test = df_area_burnt_test.copy()
##########################################################
# # BLOCK: With outliers removal - Without some outliers
# Remove outliers from Test dataset
fires_dt_test = pd.concat([fires_feat_test,df_area_burnt_test], axis=1)
#print(fires_dt_test.shape, fires_dt_test.columns)
fires_dt_test_noutlier = outlier_removal(fires_dt_test,'test')
#print(fires_dt_test_noutlier.shape, fires_dt_test_noutlier.columns)
# Recover Test dataset
X2_test = fires_dt_test_noutlier.drop(columns=['area','log_area'])
Y2_test = fires_dt_test_noutlier[['area']] 
##########################################################

# Classification Estimator
y_class_pred_TEST  = best_model_classification.predict(X2_test)

# Subset for regression: non-zero targets only
mask_non_zero_TEST  = y_class_pred_TEST == 1
X_non_zero_TEST = X2_test.loc[mask_non_zero_TEST].copy()

# Predict regression values
y_reg_pred_TEST = best_pipeline_regression_F.predict(X_non_zero_TEST)


# Step 3: Initialize full prediction array
y_full_pred_test = np.zeros_like(Y2_test)
# Step 4: Insert regression predictions into predicted non-zero locations
y_full_pred_test[mask_non_zero_TEST] = y_reg_pred_TEST


rmse = root_mean_squared_error(Y2_test, y_full_pred_test)
r2 = r2_score(Y2_test, y_full_pred_test)

print(f"Combined RMSE: {rmse:.2f}")     # Combined RMSE: 41.60
print(f"Combined R²: {r2:.2f}")         # Combined R²:   -0.09

Combined RMSE: 41.60
Combined R²: -0.09


REGRESSION ONLY STUFF, BELOW. Used and archived

In [26]:
# # Get the best pipeline from GridSearchCV, which is the fit to the whole trainiing set with the best grid parameters
# best_pipeline_B = grid_search_B.best_estimator_

# # Now access the inner pipeline (pipe_B_pre)
# inner_pipeline = best_pipeline_B.regressor_

# # Then access the model step inside that inner pipeline
# model = inner_pipeline.named_steps['model']

# # Access the regression step and grab coefficients
# coefficients = model.coef_
# intercept = model.intercept_

# print("Coefficients:", coefficients)
# print("Number of coefficients:", len(coefficients)) # Number of coefficients: 26
# print("Intercept:", intercept) # Intercept: -0.36817542069750475

In [27]:
# # Get the cross-validation results as a DataFrame
# cv_results_B = grid_search_B.cv_results_

# # Extract the mean negative MSE and r2 scores
# neg_rmse_scores = cv_results_B['mean_test_neg_rmse']

# # Convert to RMSE
# rmse_scores = (-neg_rmse_scores)

# # And if you're curious about the best RMSE, which is the average RMSE of the folds:
# best_rmse = -grid_search_B.best_score_  # grid_search.best_score_ is already neg_root_mean_squared_error, we don't need to do sqrt

# # View all RMSE scores across the grid
# for params, score in zip(cv_results_B['params'], rmse_scores):
#     print(f"Params: {params} → RMSE: {score:.2f}")
# print() 
# print("Best RMSE from CV:", best_rmse)          # Best RMSE from CV: 55.43786584769059
# print()
# print((cv_results_B.keys()))

In [28]:
# r2_scores = cv_results_B['mean_test_r2']
# for params, score in zip(cv_results_B['params'], r2_scores):
#     print(f"Params: {params} → R²: {score:.2f}")
# print()

# # Get the cross-validation results
# cv_results_B = grid_search_B.cv_results_

# # Extract the mean test R² scores
# r2_scores = cv_results_B['mean_test_r2']

# # Find the best RMSE model index
# best_index = grid_search_B.best_index_  # Index of best model (based on neg_rmse)

# # Get the corresponding R² score for that best model
# best_r2 = r2_scores[best_index]

# print("Best R² from CV:", best_r2)   # Best R² from CV: -0.05412144361174618

In [29]:
# #After Cross Validation process inside GridSearch and after assessing at the errors, if the polynomial coeff is 1, then we fit a standard LinearRegression to the whole Trainig set without folding subsets, and take those coeff.
# # Get the negative mean squared error and R2 obtained after retrained on the entire Training dataset

# # Make predictions on the full training data
# train_preds_2 = best_pipeline_B.predict(X2_train)

# # Compute RMSE root mean squared error
# train_rmse_2 = root_mean_squared_error(Y2_train, train_preds_2)

# print("RMSE on full training data:", train_rmse_2)  # RMSE in original target units     # RMSE on full training data: 68.98405061371875

# # Compute R² on the full training set
# train_r2_2 = r2_score(Y2_train, train_preds_2)

# print("R² score on full training data:", train_r2_2)                # R² score on full training data: -0.016713401138988893

In [30]:
# # RMSE and R2 of BestPipeline on Test dataset
# # Recover Test dataset
# X2_test = fires_feat_test.copy()
# Y2_test = df_area_burnt_test.copy()

# # Predict on the training or test set
# Y2_test_pred = best_pipeline_B.predict(X2_test)  # or X1_train, depending on your evaluation

# # Calculate RMSE
# rmse = root_mean_squared_error(Y2_test, Y2_test_pred)       # RMSE: 40.40514419727944

# # Calculate R-squared
# r2 = r2_score(Y2_test, Y2_test_pred)                        # R²: -0.07616305980639515

# print("RMSE:", rmse)
# print("R²:", r2)

Pipeline C

Pipeline D

# Evaluate

+ Which model has the best performance?

# Export

+ Save the best performing model to a pickle file.

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.